<h1>Chapter 10 - Creating Text Embedding Models</h1>
<i>Exploring methods for both training and fine-tuning embedding models.</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter10/Chapter%2010%20-%20Creating%20Text%20Embedding%20Models.ipynb)

---

This notebook is for Chapter 10 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [ ]:
# %%capture
# !pip install -q accelerate>=0.27.2 peft>=0.9.0 bitsandbytes>=0.43.0 transformers>=4.38.2 trl>=0.7.11 sentencepiece>=0.1.99
# !pip install -q sentence-transformers>=3.0.0 mteb>=1.1.2 datasets>=2.18.0

In [5]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Creating an Embedding Model

## **Data**

In [1]:
from datasets import load_dataset

# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

In [2]:
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

In [3]:
train_dataset[57]

{'premise': 'At the heart of the sanctuary, a small granite shrine once held the sacred barque of Horus himself.',
 'hypothesis': 'Horus is a god.',
 'label': 1}

In [4]:
train_dataset.shape

(50000, 3)

## **Model**

In [5]:
from sentence_transformers import SentenceTransformer

# Use a base model
embedding_model = SentenceTransformer('bert-base-uncased')

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.
/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## **Loss Function**

In [6]:
from sentence_transformers import losses

# Define the loss function. In soft-max loss, we will also need to explicitly set the number of labels.
train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3
)

## Evaluation

In [7]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine",
)

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

## **Training**

In [8]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="base_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

In [9]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# Train embedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/1563 [00:00<?, ?it/s]

{'loss': 1.0837, 'grad_norm': 2.746361255645752, 'learning_rate': 4.9500000000000004e-05, 'epoch': 0.06}
{'loss': 0.9589, 'grad_norm': 3.0779855251312256, 'learning_rate': 4.6616541353383456e-05, 'epoch': 0.13}
{'loss': 0.8973, 'grad_norm': 3.129072904586792, 'learning_rate': 4.3198906356801096e-05, 'epoch': 0.19}
{'loss': 0.8655, 'grad_norm': 3.0964841842651367, 'learning_rate': 3.978127136021873e-05, 'epoch': 0.26}
{'loss': 0.8414, 'grad_norm': 3.368673086166382, 'learning_rate': 3.6363636363636364e-05, 'epoch': 0.32}


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'loss': 0.8514, 'grad_norm': 4.23756742477417, 'learning_rate': 3.2946001367054005e-05, 'epoch': 0.38}
{'loss': 0.8383, 'grad_norm': 4.550880432128906, 'learning_rate': 2.9528366370471632e-05, 'epoch': 0.45}
{'loss': 0.8232, 'grad_norm': 4.936198711395264, 'learning_rate': 2.611073137388927e-05, 'epoch': 0.51}
{'loss': 0.8126, 'grad_norm': 5.204709529876709, 'learning_rate': 2.2693096377306907e-05, 'epoch': 0.58}
{'loss': 0.8027, 'grad_norm': 3.910475969314575, 'learning_rate': 1.9275461380724537e-05, 'epoch': 0.64}
{'loss': 0.7882, 'grad_norm': 3.029229164123535, 'learning_rate': 1.5857826384142175e-05, 'epoch': 0.7}
{'loss': 0.7703, 'grad_norm': 5.344746112823486, 'learning_rate': 1.2440191387559808e-05, 'epoch': 0.77}
{'loss': 0.7846, 'grad_norm': 3.518099784851074, 'learning_rate': 9.022556390977444e-06, 'epoch': 0.83}
{'loss': 0.7583, 'grad_norm': 3.2585864067077637, 'learning_rate': 5.604921394395079e-06, 'epoch': 0.9}
{'loss': 0.7902, 'grad_norm': 4.734143257141113, 'learning_r

TrainOutput(global_step=1563, training_loss=0.8417247667269911, metrics={'train_runtime': 310.0916, 'train_samples_per_second': 161.243, 'train_steps_per_second': 5.04, 'total_flos': 0.0, 'train_loss': 0.8417247667269911, 'epoch': 1.0})

In [10]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': np.float64(0.3769470904192201),
 'spearman_cosine': np.float64(0.4638190656380889),
 'pearson_manhattan': np.float64(0.4176841069073458),
 'spearman_manhattan': np.float64(0.4538681406213832),
 'pearson_euclidean': np.float64(0.3996227084827768),
 'spearman_euclidean': np.float64(0.44730404712346067),
 'pearson_dot': np.float64(0.35368057008488085),
 'spearman_dot': np.float64(0.37470393830248955),
 'pearson_max': np.float64(0.4176841069073458),
 'spearman_max': np.float64(0.4638190656380889)}

# MTEB

In [15]:
from mteb import MTEB

# Choose evaluation task
evaluation = MTEB(tasks=["Banking77Classification"])

# Calculate results
results = evaluation.run(embedding_model)
results

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

[]

⚠️ **VRAM Clean-up** - You will need to run the code below to partially empty the VRAM (GPU RAM). If that does not work, it is advised to restart the notebook instead. You can check the resources on the right-hand side (if you are using Google Colab) to check whether the used VRAM is indeed low. You can also run `!nivia-smi` to check current usage.

In [16]:
# Empty and delete trainer/model
trainer.accelerator.clear()
del trainer, embedding_model
# Garbage collection and empty cache
import gc        # Python's built-in garbage collector
import torch
gc.collect()                  # Run garbage collection
torch.cuda.empty_cache()      # Free unused GPU VRAM


# Loss Fuctions

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

## Cosine Similarity Loss

In [17]:
from datasets import Dataset, load_dataset

# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

# (neutral/contradiction)=0 and (entailment)=1
mapping = {2: 0, 1: 0, 0:1}
train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})

In [18]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [19]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="cosineloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/1563 [00:00<?, ?it/s]

{'loss': 0.2297, 'grad_norm': 1.9421889781951904, 'learning_rate': 5e-05, 'epoch': 0.06}
{'loss': 0.1723, 'grad_norm': 1.3962984085083008, 'learning_rate': 4.6582365003417636e-05, 'epoch': 0.13}
{'loss': 0.1715, 'grad_norm': 1.4664194583892822, 'learning_rate': 4.316473000683528e-05, 'epoch': 0.19}
{'loss': 0.16, 'grad_norm': 1.0665736198425293, 'learning_rate': 3.9747095010252904e-05, 'epoch': 0.26}
{'loss': 0.1527, 'grad_norm': 1.5077476501464844, 'learning_rate': 3.632946001367054e-05, 'epoch': 0.32}


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'loss': 0.1582, 'grad_norm': 1.1765071153640747, 'learning_rate': 3.291182501708818e-05, 'epoch': 0.38}
{'loss': 0.151, 'grad_norm': 1.384976863861084, 'learning_rate': 2.9494190020505813e-05, 'epoch': 0.45}
{'loss': 0.1566, 'grad_norm': 1.8200781345367432, 'learning_rate': 2.6076555023923443e-05, 'epoch': 0.51}
{'loss': 0.1509, 'grad_norm': 1.5333545207977295, 'learning_rate': 2.2658920027341084e-05, 'epoch': 0.58}
{'loss': 0.1477, 'grad_norm': 1.1415666341781616, 'learning_rate': 1.9241285030758715e-05, 'epoch': 0.64}
{'loss': 0.1475, 'grad_norm': 1.2254903316497803, 'learning_rate': 1.5823650034176352e-05, 'epoch': 0.7}
{'loss': 0.1457, 'grad_norm': 1.1953731775283813, 'learning_rate': 1.2406015037593984e-05, 'epoch': 0.77}
{'loss': 0.1457, 'grad_norm': 1.2977782487869263, 'learning_rate': 8.988380041011621e-06, 'epoch': 0.83}
{'loss': 0.1433, 'grad_norm': 1.387299656867981, 'learning_rate': 5.570745044429255e-06, 'epoch': 0.9}
{'loss': 0.1411, 'grad_norm': 1.1338657140731812, 'lea

TrainOutput(global_step=1563, training_loss=0.1576254479181896, metrics={'train_runtime': 310.9614, 'train_samples_per_second': 160.792, 'train_steps_per_second': 5.026, 'total_flos': 0.0, 'train_loss': 0.1576254479181896, 'epoch': 1.0})

In [20]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': np.float64(0.7265893782516528),
 'spearman_cosine': np.float64(0.7302613349219955),
 'pearson_manhattan': np.float64(0.7455923508174528),
 'spearman_manhattan': np.float64(0.745559000283348),
 'pearson_euclidean': np.float64(0.744896902626538),
 'spearman_euclidean': np.float64(0.7448514321793893),
 'pearson_dot': np.float64(0.6652962382230507),
 'spearman_dot': np.float64(0.6630035393963526),
 'pearson_max': np.float64(0.7455923508174528),
 'spearman_max': np.float64(0.745559000283348)}

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [21]:
#import gc        # Python's built-in garbage collector
#import torch
gc.collect()                  # Run garbage collection
torch.cuda.empty_cache()      # Free unused GPU VRAM

## Multiple Negatives Ranking Loss

In [22]:
import random
from tqdm import tqdm
from datasets import Dataset, load_dataset

# # Load MNLI dataset from GLUE
mnli = load_dataset("glue", "mnli", split="train").select(range(50_000))
mnli = mnli.remove_columns("idx")
mnli = mnli.filter(lambda x: True if x['label'] == 0 else False)

# Prepare data and add a soft negative
train_dataset = {"anchor": [], "positive": [], "negative": []}
soft_negatives = mnli["hypothesis"]
random.shuffle(soft_negatives)
for row, soft_negative in tqdm(zip(mnli, soft_negatives)):
    train_dataset["anchor"].append(row["premise"])
    train_dataset["positive"].append(row["hypothesis"])
    train_dataset["negative"].append(soft_negative)
train_dataset = Dataset.from_dict(train_dataset)
len(train_dataset)

16875it [00:00, 48152.04it/s]


16875

In [23]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [24]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/528 [00:00<?, ?it/s]

{'loss': 0.3347, 'grad_norm': 5.5606842041015625, 'learning_rate': 4.85e-05, 'epoch': 0.19}
{'loss': 0.112, 'grad_norm': 3.763345718383789, 'learning_rate': 3.866822429906542e-05, 'epoch': 0.38}
{'loss': 0.0825, 'grad_norm': 5.665317058563232, 'learning_rate': 2.698598130841122e-05, 'epoch': 0.57}
{'loss': 0.072, 'grad_norm': 1.4979768991470337, 'learning_rate': 1.530373831775701e-05, 'epoch': 0.76}
{'loss': 0.0754, 'grad_norm': 0.8218576908111572, 'learning_rate': 3.6214953271028036e-06, 'epoch': 0.95}


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'train_runtime': 128.7019, 'train_samples_per_second': 131.117, 'train_steps_per_second': 4.103, 'train_loss': 0.1315244400140011, 'epoch': 1.0}


TrainOutput(global_step=528, training_loss=0.1315244400140011, metrics={'train_runtime': 128.7019, 'train_samples_per_second': 131.117, 'train_steps_per_second': 4.103, 'total_flos': 0.0, 'train_loss': 0.1315244400140011, 'epoch': 1.0})

In [25]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': np.float64(0.8055150591517548),
 'spearman_cosine': np.float64(0.807308617630514),
 'pearson_manhattan': np.float64(0.8231464227724459),
 'spearman_manhattan': np.float64(0.8183955078439727),
 'pearson_euclidean': np.float64(0.8229086736843172),
 'spearman_euclidean': np.float64(0.8181724721028649),
 'pearson_dot': np.float64(0.72691318687068),
 'spearman_dot': np.float64(0.7153974438813474),
 'pearson_max': np.float64(0.8231464227724459),
 'spearman_max': np.float64(0.8183955078439727)}

# **Fine-tuning**

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [26]:
#import gc
#import torch

gc.collect()
torch.cuda.empty_cache()

## **Supervised**

In [27]:
from datasets import load_dataset
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [28]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="finetuned_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

  0%|          | 0/1563 [00:00<?, ?it/s]

{'loss': 0.1552, 'grad_norm': 3.5538790225982666, 'learning_rate': 5e-05, 'epoch': 0.06}
{'loss': 0.1136, 'grad_norm': 4.3015546798706055, 'learning_rate': 4.6582365003417636e-05, 'epoch': 0.13}
{'loss': 0.1196, 'grad_norm': 1.514182209968567, 'learning_rate': 4.316473000683528e-05, 'epoch': 0.19}
{'loss': 0.1127, 'grad_norm': 4.910680294036865, 'learning_rate': 3.9747095010252904e-05, 'epoch': 0.26}
{'loss': 0.1111, 'grad_norm': 5.525738716125488, 'learning_rate': 3.632946001367054e-05, 'epoch': 0.32}


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'loss': 0.0996, 'grad_norm': 3.647217035293579, 'learning_rate': 3.291182501708818e-05, 'epoch': 0.38}
{'loss': 0.1178, 'grad_norm': 4.980111122131348, 'learning_rate': 2.9494190020505813e-05, 'epoch': 0.45}
{'loss': 0.1023, 'grad_norm': 3.1382901668548584, 'learning_rate': 2.6076555023923443e-05, 'epoch': 0.51}
{'loss': 0.1011, 'grad_norm': 1.8439635038375854, 'learning_rate': 2.2658920027341084e-05, 'epoch': 0.58}
{'loss': 0.1007, 'grad_norm': 4.635411262512207, 'learning_rate': 1.9241285030758715e-05, 'epoch': 0.64}
{'loss': 0.0959, 'grad_norm': 2.5895345211029053, 'learning_rate': 1.5823650034176352e-05, 'epoch': 0.7}
{'loss': 0.1049, 'grad_norm': 1.908315658569336, 'learning_rate': 1.2406015037593984e-05, 'epoch': 0.77}
{'loss': 0.106, 'grad_norm': 3.2890660762786865, 'learning_rate': 8.988380041011621e-06, 'epoch': 0.83}
{'loss': 0.1098, 'grad_norm': 0.4102126359939575, 'learning_rate': 5.570745044429255e-06, 'epoch': 0.9}
{'loss': 0.102, 'grad_norm': 2.918778896331787, 'learnin

TrainOutput(global_step=1563, training_loss=0.1093122463384959, metrics={'train_runtime': 66.7564, 'train_samples_per_second': 748.991, 'train_steps_per_second': 23.413, 'total_flos': 0.0, 'train_loss': 0.1093122463384959, 'epoch': 1.0})

In [29]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': np.float64(0.8501711221776207),
 'spearman_cosine': np.float64(0.8492497195282527),
 'pearson_manhattan': np.float64(0.8512479539005469),
 'spearman_manhattan': np.float64(0.8482876868358741),
 'pearson_euclidean': np.float64(0.8522362466406264),
 'spearman_euclidean': np.float64(0.8492497195282527),
 'pearson_dot': np.float64(0.8501711241189458),
 'spearman_dot': np.float64(0.8492497195282527),
 'pearson_max': np.float64(0.8522362466406264),
 'spearman_max': np.float64(0.8492497195282527)}

In [30]:
# Evaluate the pre-trained model
original_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
evaluator(original_model)

/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'pearson_cosine': np.float64(0.869619461548402),
 'spearman_cosine': np.float64(0.8671631197908374),
 'pearson_manhattan': np.float64(0.8670399065811152),
 'spearman_manhattan': np.float64(0.8663946139224048),
 'pearson_euclidean': np.float64(0.8678716027448885),
 'spearman_euclidean': np.float64(0.8671631197908374),
 'pearson_dot': np.float64(0.8696194640672229),
 'spearman_dot': np.float64(0.8671631197908374),
 'pearson_max': np.float64(0.8696194640672229),
 'spearman_max': np.float64(0.8671631197908374)}

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [31]:
#import gc
#import torch

gc.collect()
torch.cuda.empty_cache()

## **Augmented SBERT**

**Step 1:** Fine-tune a cross-encoder

In [6]:
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import InputExample
from sentence_transformers.datasets import NoDuplicatesDataLoader

# Prepare a small set of 10000 documents for the cross-encoder
dataset = load_dataset("glue", "mnli", split="train").select(range(10_000))
mapping = {2: 0, 1: 0, 0:1}

# Data Loader
gold_examples = [
    InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
    for row in tqdm(dataset)
]
gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)

# Pandas DataFrame for easier data handling
gold = pd.DataFrame(
    {
    'sentence1': dataset['premise'],
    'sentence2': dataset['hypothesis'],
    'label': [mapping[label] for label in dataset['label']]
    }
)

100%|██████████| 10000/10000 [00:00<00:00, 77001.39it/s]


In [7]:
from sentence_transformers.cross_encoder import CrossEncoder

# Train a cross-encoder on the gold dataset
cross_encoder = CrossEncoder('bert-base-uncased', num_labels=2)
cross_encoder.fit(
    train_dataloader=gold_dataloader,
    epochs=1,
    show_progress_bar=True,
    warmup_steps=100,
    use_amp=False
)

/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/312 [00:00<?, ?it/s]

**Step 2:** Create new sentence pairs

In [8]:
# Prepare the silver dataset by predicting labels with the cross-encoder
silver = load_dataset("glue", "mnli", split="train").select(range(10_000, 50_000))
pairs = list(zip(silver['premise'], silver['hypothesis']))

**Step 3:** Label new sentence pairs with the fine-tuned cross-encoder (silver dataset)

In [9]:
import numpy as np

# Label the sentence pairs using our fine-tuned cross-encoder
output = cross_encoder.predict(pairs, apply_softmax=True, show_progress_bar=True)
silver = pd.DataFrame(
    {
        "sentence1": silver["premise"],
        "sentence2": silver["hypothesis"],
        "label": np.argmax(output, axis=1)
    }
)

Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

**Step 4:** Train a bi-encoder (SBERT) on the extended dataset (gold + silver dataset)

In [10]:
# Combine gold + silver
data = pd.concat([gold, silver], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=['sentence1', 'sentence2'], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

In [11]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [12]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="augmented_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.
/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/1563 [00:00<?, ?it/s]

{'loss': 0.2131, 'grad_norm': 1.6065479516983032, 'learning_rate': 5e-05, 'epoch': 0.06}
{'loss': 0.1572, 'grad_norm': 1.4884434938430786, 'learning_rate': 4.6582365003417636e-05, 'epoch': 0.13}
{'loss': 0.1436, 'grad_norm': 1.7958072423934937, 'learning_rate': 4.316473000683528e-05, 'epoch': 0.19}
{'loss': 0.1419, 'grad_norm': 1.2754000425338745, 'learning_rate': 3.9747095010252904e-05, 'epoch': 0.26}
{'loss': 0.1366, 'grad_norm': 1.162861943244934, 'learning_rate': 3.632946001367054e-05, 'epoch': 0.32}


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'loss': 0.14, 'grad_norm': 1.4196408987045288, 'learning_rate': 3.291182501708818e-05, 'epoch': 0.38}
{'loss': 0.1327, 'grad_norm': 1.2426141500473022, 'learning_rate': 2.9494190020505813e-05, 'epoch': 0.45}
{'loss': 0.1311, 'grad_norm': 1.206871747970581, 'learning_rate': 2.6076555023923443e-05, 'epoch': 0.51}
{'loss': 0.1315, 'grad_norm': 1.0839767456054688, 'learning_rate': 2.2658920027341084e-05, 'epoch': 0.58}
{'loss': 0.1276, 'grad_norm': 1.331589937210083, 'learning_rate': 1.9241285030758715e-05, 'epoch': 0.64}
{'loss': 0.1275, 'grad_norm': 1.2218385934829712, 'learning_rate': 1.5823650034176352e-05, 'epoch': 0.7}
{'loss': 0.129, 'grad_norm': 0.8450924158096313, 'learning_rate': 1.2406015037593984e-05, 'epoch': 0.77}
{'loss': 0.1282, 'grad_norm': 0.9308277368545532, 'learning_rate': 8.988380041011621e-06, 'epoch': 0.83}
{'loss': 0.1259, 'grad_norm': 1.4188905954360962, 'learning_rate': 5.570745044429255e-06, 'epoch': 0.9}
{'loss': 0.1281, 'grad_norm': 0.9897803664207458, 'learn

TrainOutput(global_step=1563, training_loss=0.13896672571613022, metrics={'train_runtime': 297.726, 'train_samples_per_second': 167.933, 'train_steps_per_second': 5.25, 'total_flos': 0.0, 'train_loss': 0.13896672571613022, 'epoch': 1.0})

In [13]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': np.float64(0.6692431026513692),
 'spearman_cosine': np.float64(0.6893066020323626),
 'pearson_manhattan': np.float64(0.7022089963891431),
 'spearman_manhattan': np.float64(0.700803138435996),
 'pearson_euclidean': np.float64(0.7009543825554414),
 'spearman_euclidean': np.float64(0.6998182124893843),
 'pearson_dot': np.float64(0.6216860746486439),
 'spearman_dot': np.float64(0.6261465157402865),
 'pearson_max': np.float64(0.7022089963891431),
 'spearman_max': np.float64(0.700803138435996)}

In [14]:
trainer.accelerator.clear()

[]

**Step 5**: Evaluate without silver dataset

In [15]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()


In [16]:
# Combine gold + silver
data = pd.concat([gold], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=['sentence1', 'sentence2'], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="gold_only_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.
/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/312 [00:00<?, ?it/s]

{'loss': 0.2254, 'grad_norm': 1.2423127889633179, 'learning_rate': 5e-05, 'epoch': 0.32}
{'loss': 0.171, 'grad_norm': 1.5132402181625366, 'learning_rate': 2.641509433962264e-05, 'epoch': 0.64}
{'loss': 0.1601, 'grad_norm': 1.3765714168548584, 'learning_rate': 2.830188679245283e-06, 'epoch': 0.96}
{'train_runtime': 84.2812, 'train_samples_per_second': 118.65, 'train_steps_per_second': 3.702, 'train_loss': 0.1848368117442498, 'epoch': 1.0}


TrainOutput(global_step=312, training_loss=0.1848368117442498, metrics={'train_runtime': 84.2812, 'train_samples_per_second': 118.65, 'train_steps_per_second': 3.702, 'total_flos': 0.0, 'train_loss': 0.1848368117442498, 'epoch': 0.9984})

In [17]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': np.float64(0.6684507130506666),
 'spearman_cosine': np.float64(0.6885029324385956),
 'pearson_manhattan': np.float64(0.6821378454410685),
 'spearman_manhattan': np.float64(0.6901200936969646),
 'pearson_euclidean': np.float64(0.6820452128576884),
 'spearman_euclidean': np.float64(0.690068202202079),
 'pearson_dot': np.float64(0.5933672460835088),
 'spearman_dot': np.float64(0.5917312713493896),
 'pearson_max': np.float64(0.6821378454410685),
 'spearman_max': np.float64(0.6901200936969646)}

Compared to using both the silver and gold datasets, using only the gold dataset reduces the performance of the model!

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [18]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## **Unsupervised Learning**

### Tranformer-based Denoising AutoEncoder (TSDAE)

In [19]:
# Download additional tokenizer
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/cairo/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [20]:
from tqdm import tqdm
from datasets import Dataset, load_dataset
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

# Create a flat list of sentences
mnli = load_dataset("glue", "mnli", split="train").select(range(25_000))
flat_sentences = mnli["premise"] + mnli["hypothesis"]

# Add noise to our input data
damaged_data = DenoisingAutoEncoderDataset(list(set(flat_sentences)))

# Create dataset
train_dataset = {"damaged_sentence": [], "original_sentence": []}
for data in tqdm(damaged_data):
    train_dataset["damaged_sentence"].append(data.texts[0])
    train_dataset["original_sentence"].append(data.texts[1])
train_dataset = Dataset.from_dict(train_dataset)

100%|██████████| 48353/48353 [00:03<00:00, 13022.06it/s]


In [21]:
train_dataset[0]

{'damaged_sentence': 'i do a really delicate moral issue guess do that be a solution if into country have a certain obligation',
 'original_sentence': "i i don't either it's a really delicate uh moral issue because if if you have well i guess the one thing i do see that is is that needs to be a solution is that if you do let people into the country i do feel you have a certain obligation"}

In [22]:
# # Choose a different deletion ratio
# flat_sentences = list(set(flat_sentences))
# damaged_data = DenoisingAutoEncoderDataset(
#     flat_sentences,
#     noise_fn=lambda s: DenoisingAutoEncoderDataset.delete(s, del_ratio=0.6)
# )

In [23]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [24]:
from sentence_transformers import models, SentenceTransformer

# Create your embedding model
word_embedding_model = models.Transformer('bert-base-uncased')
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), 'cls')
embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [25]:
from sentence_transformers import losses

# Use the denoising auto-encoder loss
train_loss = losses.DenoisingAutoEncoderLoss(
    embedding_model, tie_encoder_decoder=True
)
train_loss.decoder = train_loss.decoder.to("cuda")

Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.self.key.bias', 'bert.e

In [26]:
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="tsdae_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/3023 [00:00<?, ?it/s]

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


{'loss': 7.102, 'grad_norm': 8.127017974853516, 'learning_rate': 4.9e-05, 'epoch': 0.03}
{'loss': 4.9244, 'grad_norm': 6.3062615394592285, 'learning_rate': 4.8323640095791995e-05, 'epoch': 0.07}
{'loss': 4.6609, 'grad_norm': 7.259078502655029, 'learning_rate': 4.66130687649675e-05, 'epoch': 0.1}
{'loss': 4.5547, 'grad_norm': 7.425272464752197, 'learning_rate': 4.4902497434143e-05, 'epoch': 0.13}
{'loss': 4.4956, 'grad_norm': 6.683529376983643, 'learning_rate': 4.319192610331851e-05, 'epoch': 0.17}


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'loss': 4.4156, 'grad_norm': 6.6163153648376465, 'learning_rate': 4.1481354772494014e-05, 'epoch': 0.2}
{'loss': 4.3388, 'grad_norm': 6.684187412261963, 'learning_rate': 3.977078344166952e-05, 'epoch': 0.23}
{'loss': 4.2917, 'grad_norm': 6.445783615112305, 'learning_rate': 3.806021211084502e-05, 'epoch': 0.26}
{'loss': 4.2513, 'grad_norm': 6.904947757720947, 'learning_rate': 3.634964078002053e-05, 'epoch': 0.3}
{'loss': 4.2359, 'grad_norm': 6.693707466125488, 'learning_rate': 3.463906944919603e-05, 'epoch': 0.33}
{'loss': 4.2283, 'grad_norm': 6.477916240692139, 'learning_rate': 3.2928498118371536e-05, 'epoch': 0.36}
{'loss': 4.1392, 'grad_norm': 6.30422830581665, 'learning_rate': 3.121792678754704e-05, 'epoch': 0.4}
{'loss': 4.1282, 'grad_norm': 7.319807529449463, 'learning_rate': 2.9507355456722545e-05, 'epoch': 0.43}
{'loss': 4.1355, 'grad_norm': 6.923688888549805, 'learning_rate': 2.7796784125898052e-05, 'epoch': 0.46}
{'loss': 4.0683, 'grad_norm': 7.012391090393066, 'learning_rate

TrainOutput(global_step=3023, training_loss=4.2341770849754745, metrics={'train_runtime': 596.5635, 'train_samples_per_second': 81.053, 'train_steps_per_second': 5.067, 'total_flos': 0.0, 'train_loss': 4.2341770849754745, 'epoch': 1.0})

In [27]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': np.float64(0.6973720507415031),
 'spearman_cosine': np.float64(0.7125488404920083),
 'pearson_manhattan': np.float64(0.7067307365368829),
 'spearman_manhattan': np.float64(0.7109088482570294),
 'pearson_euclidean': np.float64(0.7055151453177229),
 'spearman_euclidean': np.float64(0.7098350877295387),
 'pearson_dot': np.float64(0.5750846105093801),
 'spearman_dot': np.float64(0.5783298065848816),
 'pearson_max': np.float64(0.7067307365368829),
 'spearman_max': np.float64(0.7125488404920083)}

In [28]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()